In [ ]:
!python -m pip install -U python-woc pandas matplotlib

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'microsoft/TextWorld',
    'microsoft/unilm',
    'ExaScience/macau',
    'keras-team/autokeras',
    'cgentemann/cloud_science',
    'gauzias/slam',
    'Ivanlh20/multem',
    'slundberg/shap',
    'MITgcm/MITgcm',
    'MrGiovanni/ModelsGenesis'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00011bd91daf73b71f1278d52a35424b176a8807,microsoft_textworld


In [5]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  0%|▏                                                                                                                                                                                                          | 5/4682 [00:05<1:19:31,  1.02s/it]

Got Errors {'09119a40fb6362614aade31732bed8b8d15eefff': 'Key 09119a40fb6362614aade31732bed8b8d15eefff not found in /da5_fast/All.sha1c/commit_9.tch', '09bfc8e053b6c2ccf28597e6c6872c6cb45fa31c': 'Key 09bfc8e053b6c2ccf28597e6c6872c6cb45fa31c not found in /da5_fast/All.sha1c/commit_9.tch'}


  0%|▎                                                                                                                                                                                                          | 8/4682 [00:08<1:19:43,  1.02s/it]

Got Errors {'1272d4baddca7f942e61341c02762ba013265e26': 'Key 1272d4baddca7f942e61341c02762ba013265e26 not found in /da5_fast/All.sha1c/commit_18.tch'}


  0%|▍                                                                                                                                                                                                         | 10/4682 [00:10<1:19:32,  1.02s/it]

...

Got Errors {'f3d853ee7e172d2320bdb7099c54b3f4557a625a': 'Key f3d853ee7e172d2320bdb7099c54b3f4557a625a not found in /da5_fast/All.sha1c/commit_115.tch'}


 98%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏   | 4594/4682 [1:18:58<01:30,  1.03s/it]

Got Errors {'fa4bb9c7e8f1771276df76967c0aaae2713bf985': 'Key fa4bb9c7e8f1771276df76967c0aaae2713bf985 not found in /da5_fast/All.sha1c/commit_122.tch'}


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4682/4682 [1:20:29<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,00011bd91daf73b71f1278d52a35424b176a8807,cfa8d062707922f663311f5a9c1f541364611d15,[d92d14eb861bba2095258f49f054402303ea8d18],Marc-Alexandre Côté <marc.cote.19@gmail.com>,1663382841,-0400,Marc-Alexandre Côté <marc.cote.19@gmail.com>,1663382922,-0400,Rename main branch and references to it.\n,00011bd91daf73b71f1278d52a35424b176a8807,microsoft_textworld
1,00640623ba2c422d75e3d23eed92d24876037ade,d22a175b1f4b62384f79558fe2e113af6519d68d,[0593109adec574e5a72c7d3cd7f61ea816006767],Marc-Alexandre Côté <marc.cote.19@gmail.com>,1702342627,-0500,GitHub <noreply@github.com>,1702342627,-0500,Update prerelease.yml,00640623ba2c422d75e3d23eed92d24876037ade,microsoft_textworld


In [6]:
print("Retrieved commits:", len(commit_data))
print("Total SHA-1s:", len(df))
print("Missing commits:", len(df) - len(commit_data))

Retrieved commits: 46711
Total SHA-1s: 46816
Missing commits: 105


In [7]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '00011bd91daf73b71f1278d52a35424b176a8807', 'tree': 'cfa8d062707922f663311f5a9c1f541364611d15', 'parent': ['d92d14eb861bba2095258f49f054402303ea8d18'], 'author': 'Marc-Alexandre Côté <marc.cote.19@gmail.com>', 'author_time': 1663382841, 'author_tz': '-0400', 'committer': 'Marc-Alexandre Côté <marc.cote.19@gmail.com>', 'committer_time': 1663382922, 'committer_tz': '-0400', 'message': 'Rename main branch and references to it.\n'}


In [8]:
# Keep each commit associated with its correct project.
dfinf = df_commit_data[
    ['project', 'commit', 'author', 'author_time', 'message']
].copy()

dfinf.columns = [
    'project_wocid',
    'commit_sha1',
    'author',
    'time',
    'commit message'
]

dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,microsoft_textworld,00011bd91daf73b71f1278d52a35424b176a8807,Marc-Alexandre Côté <marc.cote.19@gmail.com>,1663382841,Rename main branch and references to it.\n


In [9]:
# Check that the CSV has the required columns and content.
dfinf.head(1)

,project_wocid,commit_sha1,author,time,commit message
0,microsoft_textworld,00011bd91daf73b71f1278d52a35424b176a8807,Marc-Alexandre Côté <marc.cote.19@gmail.com>,1663382841,Rename main branch and references to it.\n


In [10]:
# Save one semicolon-separated CSV containing all retrieved projects.
yournetid = 'smazunda'
dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';')

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [11]:
# Report WoC commit counts, unique authors, and timestamp range for each project.
df_commit_data['author_time'] = pd.to_numeric(df_commit_data['author_time'], errors='coerce')
woc_stats = df_commit_data.groupby('project').agg(
    ncommits=('commit', 'nunique'),
    nauthors=('author', 'nunique'),
    min_time=('author_time', 'min'),
    max_time=('author_time', 'max')
).reset_index()

woc_stats['min_time'] = pd.to_datetime(woc_stats['min_time'], unit='s', utc=True)
woc_stats['max_time'] = pd.to_datetime(woc_stats['max_time'], unit='s', utc=True)
display(woc_stats)

,project,ncommits,nauthors,min_time,max_time
0,cgentemann_cloud_science,574,16,2019-04-01 21:40:04+00:00,2024-11-07 17:39:46+00:00
1,exascience_macau,4273,17,2016-01-05 11:36:25+00:00,2025-03-10 13:32:05+00:00
2,gauzias_slam,596,27,2019-04-30 08:27:53+00:00,2024-11-29 10:34:15+00:00
3,ivanlh20_multem,420,20,2014-08-29 14:53:07+00:00,2025-09-14 08:44:06+00:00
4,keras-team_autokeras,5202,273,2017-11-19 23:18:21+00:00,2025-01-14 08:45:02+00:00
5,microsoft_textworld,927,44,2018-06-26 18:00:56+00:00,2025-01-23 17:19:27+00:00
6,microsoft_unilm,1578,161,2019-07-23 04:15:29+00:00,2025-09-13 14:02:31+00:00
7,mitgcm_mitgcm,26776,206,1998-04-22 19:15:31+00:00,2025-11-08 17:31:14+00:00
8,mrgiovanni_modelsgenesis,274,15,2019-07-24 19:07:20+00:00,2025-06-22 18:38:35+00:00
9,slundberg_shap,6091,588,2016-11-22 21:44:11+00:00,2025-11-12 02:59:12+00:00


In [13]:
# Retrieve GitHub stars, forks, and latest commit date for each repository.
import requests
import time

github_rows = []
for repo in projects:
    response = requests.get(f'https://api.github.com/repos/{repo}', timeout=30)
    response.raise_for_status()
    repo_data = response.json()

    commit_response = requests.get(
        f'https://api.github.com/repos/{repo}/commits',
        params={'per_page': 1},
        timeout=30
    )
    commit_response.raise_for_status()
    commit_list = commit_response.json()

    github_rows.append({
        'repository': repo,
        'stars': repo_data['stargazers_count'],
        'forks': repo_data['forks_count'],
        'last_commit_date': (
            commit_list[0]['commit']['committer']['date']
            if commit_list else 'N/A'
        )
    })
    time.sleep(1)

github_stats = pd.DataFrame(github_rows)
display(github_stats)

,repository,stars,forks,last_commit_date
0,microsoft/TextWorld,1447,206,2026-09-28T13:36:06Z
1,microsoft/unilm,22225,2706,2026-09-21T03:26:59Z
2,ExaScience/macau,69,12,2026-02-10T07:25:29Z
3,keras-team/autokeras,9333,1388,2025-11-25T02:26:35Z
4,cgentemann/cloud_science,26,17,2024-11-07T17:39:46Z
5,gauzias/slam,1,0,2024-11-29T10:31:33Z
6,Ivanlh20/multem,80,29,2025-07-03T14:04:49Z
7,slundberg/shap,25784,3755,2026-09-28T18:57:07Z
8,MITgcm/MITgcm,407,296,2026-09-06T15:41:07Z
9,MrGiovanni/ModelsGenesis,794,142,2026-08-25T17:16:33Z
